### Structured Output ###

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic - Pi Dent Tak

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000214E5B66510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000214E5B66F90>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="This title of the movie")
    description: str = Field(description="This movie description")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="the movie's rating out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.5', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000016BDE1230E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000016BDE123B60>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'This t

In [4]:
# This is how we get default output of movie
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception (2010) – Quick Reference**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Date** | July\u202f16\u202f2010 (United States) |\n| **Running Time** | 148 minutes |\n| **Genre** | Science‑fiction, Action, Thriller |\n| **Director / Writer** | Christopher Nolan |\n| **Production Companies** | Warner Bros. Pictures, Legendary Pictures, Syncopy Inc. |\n| **Budget** | ≈\u202f$160\u202fmillion |\n| **Box‑Office Gross** | ≈\u202f$839\u202fmillion worldwide (≈\u202f$292\u202fM domestic, $547\u202fM international) |\n| **MPAA Rating** | PG‑13 (Violence, some language, brief strong sexual content) |\n| **Filming Locations** | Los Angeles (CA), Tokyo (Japan), Paris (France), London (UK), Calgary (Canada), Morocco, United Kingdom (various studios) |\n| **Music** | Hans Zimmer (original score) |\n| **Cinematography** | Wally Pfister (Academy Award for Best Cinematography) |\n| **Editing** | Lee Smith |\n| **Production Designer** | Guy 

In [13]:
# This is how we get Structured output of movie
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', description="A skilled thief, Dom Cobb, who specializes in extracting secrets from deep within the subconscious during the dream state, is offered a chance to have his criminal record erased if he can successfully perform an 'inception'—planting an idea into a target's mind. He assembles a team and navigates layered dream worlds, confronting his own past and the blurred line between reality and illusion.", year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside parsed strucutre

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """ A movie with details """
    title: str = Field(..., description="This title of the movie")
    description: str = Field(..., description="This movie description like 10 character after that ...")
    year: int = Field(..., description="This year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="the movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie "Inception". Use the function Movie with fields: description, director, rating, title, year. Provide description limited to about 10 characters? The function comment: "This movie description like 10 character after that ..." unclear. Probably they want a short description maybe 10 characters? Might be a typo. We can give a brief description. Use function.', 'tool_calls': [{'id': 'fc_7214fe1c-261b-4d68-a481-622ed2767ee6', 'function': {'arguments': '{"description":"Dream heist thriller","director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 177, 'total_tokens': 321, 'completion_time': 0.306624018, 'completion_tokens_details': {'reasoning_tokens': 82}, 'prompt_time': 0.093507859, 'prompt_tokens_details': None, 'queue_time': 0.355597801,

### Nested Structure

In [16]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Pete Postlethwaite', role='Robert Fischer Sr.')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

### TypedDict

TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [19]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """ A movie with detils."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year of the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypeddict = model.with_structured_output(MovieDict)
response = model_withtypeddict.invoke("Pelease provide the details of the movie avengers")

response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [22]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None  = Annotated[None, ..., "Budget in millions USD"]

model_withtypeddict = model.with_structured_output(MovieDetails)

response = model_withtypeddict.invoke("Provide details about the movie Inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'},
  {'name': 'Marion Cotillard', 'role': 'Mal'},
  {'name': 'Michael Caine', 'role': 'Professor Stephen Miles'},
  {'name': 'Dileep Rao', 'role': 'Yusuf'},
  {'name': 'Tom Berenger', 'role': 'Peter Browning'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [25]:
# model_withtypeddict.profile
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### Data Classes

A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [3]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """ Contact information for a person. """
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
print(result["structured_response"].name)
print(result["structured_response"].email)
print(result["structured_response"].phone)
result

name='John Doe' email='john@example.com' phone='(555) 123-4567'
John Doe
john@example.com
(555) 123-4567


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='4b5d9416-4e7f-459a-a6e1-aca2cefc6ee0'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'We need to output JSON according to ContactInfo schema: fields name, email, phone required. Provide compact JSON.\n\nExtracted: name "John Doe", email "john@example.com", phone "(555) 123-4567". Ensure proper JSON.\n\nWe output only JSON object.'}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 239, 'total_tokens': 333, 'completion_time': 0.195476087, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.01493895, 'prompt_tokens_details': None, 'queue_time': 0.400870233, 'total_time': 0.210415037}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_93703442d9', 'service_tier': 'on_demand', 'finish_rea

In [37]:
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

# class ContactInfo(TypedDict):
#     """ Contact information for a person. """
#     name: Annotated[str, ..., "The name of the person"]
#     email: Annotated[str, ..., "The email address of the person"]
#     phone: Annotated[str, ..., "The phone number of the person"]

#  ----------------- OR ------------------------

class ContactInfo(TypedDict):
    """ Contact information for a person. """
    name: str
    email: str
    phone: str

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages" : [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
print(result["structured_response"]["name"])
print(result["structured_response"]["email"])
print(result["structured_response"]["phone"])
result

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}
John Doe
john@example.com
(555) 123-4567


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='21201674-e37f-42b2-8d28-66fb14aa86c9'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info from the given text. The system says we need to output only the final JSON object, adhering to the ContactInfo schema. Must include name, email, phone. Provide compact JSON (no whitespace). So output: {"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"} Ensure valid JSON.'}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 213, 'total_tokens': 326, 'completion_time': 0.237777745, 'completion_tokens_details': {'reasoning_tokens': 78}, 'prompt_time': 0.008470294, 'prompt_tokens_details': None, 'queue_time': 0.216593043, 'total_time': 0.246248039}, 'model_name': 'openai/gpt-oss

In [5]:
### Now by using DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ Contact information for a person. """
    name: str
    email: str
    phone: str

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

response = agent.invoke({
    "messages" : [{
        "role": "user",
        "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567 ", 
    }]
})

response["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')